In [2]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
import joblib

In [3]:
DATA_DIR = "Images" 
IMG_SIZE = (64, 64)         
SAMPLES_PER_CLASS = 1500     
RANDOM_STATE = 42

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
def find_image_paths(data_dir):
    exts = ("*.jpg", "*.jpeg", "*.png")
    items = []

    # Case-insensitive lookup for cat/dog subfolders
    cat_dir = dog_dir = None
    if os.path.isdir(data_dir):
        for entry in os.listdir(data_dir):
            full = os.path.join(data_dir, entry)
            if os.path.isdir(full):
                if entry.lower() == "cat":
                    cat_dir = full
                elif entry.lower() == "dog":
                    dog_dir = full

    if cat_dir and dog_dir:
        for ext in exts:
            for p in glob.glob(os.path.join(cat_dir, ext)) + glob.glob(os.path.join(cat_dir, ext.upper())):
                items.append((p, 0))
            for p in glob.glob(os.path.join(dog_dir, ext)) + glob.glob(os.path.join(dog_dir, ext.upper())):
                items.append((p, 1))
    else:
        for ext in exts:
            for p in glob.glob(os.path.join(data_dir, "**", ext), recursive=True):
                fname = os.path.basename(p).lower()
                if "cat" in fname:
                    items.append((p, 0))
                elif "dog" in fname:
                    items.append((p, 1))

    return items

In [5]:
def extract_hog_features(img_path, img_size=IMG_SIZE):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, img_size)
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )
    return features


def build_dataset(items, samples_per_class=SAMPLES_PER_CLASS):
    cats = [i for i in items if i[1] == 0][:samples_per_class]
    dogs = [i for i in items if i[1] == 1][:samples_per_class]
    subset = cats + dogs

    X, y = [], []
    for path, label in subset:
        feat = extract_hog_features(path)
        if feat is not None:
            X.append(feat)
            y.append(label)

    return np.array(X), np.array(y)


In [6]:
def main():
    print(f"Scanning {DATA_DIR} for images...")
    items = find_image_paths(DATA_DIR)
    n_cat = sum(1 for _, l in items if l == 0)
    n_dog = sum(1 for _, l in items if l == 1)
    print(f"Found {n_cat} cat images, {n_dog} dog images.")

    if n_cat == 0 or n_dog == 0:
        raise FileNotFoundError(
            f"No images found under {DATA_DIR}. "
            "Expected 'cat/' and 'dog/' subfolders, or filenames containing "
            "'cat'/'dog'. Update DATA_DIR at the top of the script."
        )

    print(f"Extracting HOG features (using up to {SAMPLES_PER_CLASS} per class)...")
    X, y = build_dataset(items)
    print(f"Feature matrix shape: {X.shape}")

    # ---------- Train/test split ----------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # ---------- Scale features ----------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ---------- Dimensionality reduction (PCA) ----------
    pca = PCA(n_components=150, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    print(f"PCA: reduced to {pca.n_components_} components "
          f"({pca.explained_variance_ratio_.sum():.2%} variance retained)")

    # ---------- Hyperparameter search ----------
    print("\nRunning grid search for best SVM hyperparameters...")
    param_grid = {
        "C": [1, 10, 100],
        "gamma": ["scale", 0.01, 0.001],
        "kernel": ["rbf"],
    }
    grid = GridSearchCV(
        SVC(class_weight="balanced"),
        param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train_pca, y_train)
    print(f"Best params: {grid.best_params_}")
    print(f"Best CV accuracy: {grid.best_score_:.4f}")

    best_svm = grid.best_estimator_

    # ---------- Evaluate ----------
    y_pred = best_svm.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n=== Test Set Performance ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))

    # ---------- Confusion matrix ----------
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cat", "Dog"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title("SVM Confusion Matrix — Cat vs Dog")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
    plt.close()

    # ---------- Save model + preprocessing pipeline ----------
    joblib.dump(
        {"svm": best_svm, "scaler": scaler, "pca": pca, "img_size": IMG_SIZE},
        os.path.join(OUT_DIR, "svm_cat_dog_model.joblib"),
    )
    print(f"\nSaved model pipeline to {OUT_DIR}/svm_cat_dog_model.joblib")
    print(f"Saved confusion matrix to {OUT_DIR}/confusion_matrix.png")

In [7]:
def predict_image(image_path, model_path=os.path.join(OUT_DIR, "svm_cat_dog_model.joblib")):
    bundle = joblib.load(model_path)
    feat = extract_hog_features(image_path, bundle["img_size"]).reshape(1, -1)
    feat_scaled = bundle["scaler"].transform(feat)
    feat_pca = bundle["pca"].transform(feat_scaled)
    pred = bundle["svm"].predict(feat_pca)[0]
    return "Dog" if pred == 1 else "Cat"

In [8]:
if __name__ == "__main__":
    main()

Scanning Images for images...
Found 25000 cat images, 25000 dog images.
Extracting HOG features (using up to 1500 per class)...
Feature matrix shape: (2999, 1764)
PCA: reduced to 150 components (66.63% variance retained)

Running grid search for best SVM hyperparameters...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best params: {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
Best CV accuracy: 0.7453

=== Test Set Performance ===
Accuracy: 0.7700

Classification Report:
              precision    recall  f1-score   support

         Cat       0.78      0.75      0.77       300
         Dog       0.76      0.79      0.77       300

    accuracy                           0.77       600
   macro avg       0.77      0.77      0.77       600
weighted avg       0.77      0.77      0.77       600


Saved model pipeline to outputs/svm_cat_dog_model.joblib
Saved confusion matrix to outputs/confusion_matrix.png


In [9]:
predict_image(image_path="test images/image1.jpg")

'Cat'

In [11]:
predict_image(image_path="test images/image2.jpg")

'Dog'